# SCAMPER Demo

This notebook allows users to rollout and render SCAMPER and its baselines (MIMIC-MJX and the from-scratch model) on all the tasks presented in *Sirbu et al. 2026*

Unless the user has access to a 80GB GPU, it is recommended that rollouts are loaded from the provided data and rendered rather than rolling out the tasks themselves (i.e., skip the cells marked ROLLOUT)


In [ ]:
import os

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import gc

import jax
from IPython.display import Video, display

from scamper.scripts.collect_rollout_dataset import (
    create_behavior_prior_policy_fn,
    create_prior_policy_fn as create_motion_prior_policy_fn,
)
from scamper.utils.demo_utils import *


def release(*names):
    """Drop named globals, then gc.collect() and jax.clear_caches()."""
    g = globals()
    for n in names:
        if n in g:
            try:
                del g[n]
            except KeyError:
                pass
    gc.collect()
    try:
        jax.clear_caches()
    except Exception:
        pass
    print(f"released {list(names)}; jax live arrays: {len(jax.live_arrays())}")

In [ ]:
# ============ STORAGE PATHS ============
DEMO_ROOT = Path("../data/demos")
ROLLOUTS_DIR = DEMO_ROOT / "rollouts"
VIDEOS_DIR = DEMO_ROOT / "videos"
ROLLOUTS_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# ============ CLIP SELECTION ============
CLIP_INDICES = [0, 2, 3, 4, 5, 6, 8, 11, 12]  # behavior_prior + motion_prior grids
NUM_CLIPS = len(CLIP_INDICES)
IMITATION_CLIP_INDICES = [0, 2, 3, 4, 5, 6, 8, 11, 12]  # imitation grid only
IMITATION_NUM_CLIPS = len(IMITATION_CLIP_INDICES)

# ============ RUN CONFIG ============
NUM_ROLLOUT_STEPS = 1000
SEED = 42

# ============ CHECKPOINTS — FILL IN ============
# Section 1 (imitation): a distill checkpoint whose teacher is the imitation policy.
IMITATION_DISTILL_CHECKPOINT = "../data/checkpoints/scamper/stage2/260327_134740_079934"
IMITATION_DISTILL_CHECKPOINT_STEP = None  # None = latest
IMITATION_TEACHER_CHECKPOINT_PATH = (
    None  # None = restore teacher from inside the distill checkpoint
)
IMITATION_TEACHER_CHECKPOINT_STEP = None

# Sections 2/3/4 (behavior prior + motion prior): a distill_motion_prior checkpoint that contains behavior_prior + motion_prior + decoder.
DISTILL_MOTION_PRIOR_CHECKPOINT = (
    "../data/checkpoints/scamper/stage3/260406_130524_381481"
)
DISTILL_MOTION_PRIOR_CHECKPOINT_STEP = None  # None = latest

# Sections 5/6/7 (walk-rear). Modes: "res_control", "mimic_mjx", "scratch".
WALKREAR_SCAMPER_CHECKPOINT = (
    "../data/checkpoints/walk_rear/scamper/260415_195913_559341"
)
WALKREAR_SCAMPER_STEP = None
WALKREAR_MIMICMJX_CHECKPOINT = (
    "../data/checkpoints/walk_rear/mimic-mjx/260415_193318_969191"
)
WALKREAR_MIMICMJX_STEP = None
WALKREAR_SCRATCH_CHECKPOINT = (
    "../data/checkpoints/walk_rear/scratch/260415_194114_644658"
)
WALKREAR_SCRATCH_STEP = None
WALKREAR_NUM_STEPS = 500

# Sections 8/9/10 (arena-forage). Modes: "prior_gating", "mimic_mjx", "scratch".
ARENAFORAGE_SCAMPER_CHECKPOINT = "../data/checkpoints/arena_forage/scamper_forage"
ARENAFORAGE_SCAMPER_STEP = None
ARENAFORAGE_MIMICMJX_CHECKPOINT = "../data/checkpoints/arena_forage/mimic-mjx_forage"
ARENAFORAGE_MIMICMJX_STEP = None
ARENAFORAGE_SCRATCH_CHECKPOINT = "../data/checkpoints/arena_forage/scratch_forage"
ARENAFORAGE_SCRATCH_STEP = None
ARENAFORAGE_NUM_STEPS = 1500

print(f"Demo root: {DEMO_ROOT}")
print(f"  rollouts -> {ROLLOUTS_DIR}")
print(f"  videos   -> {VIDEOS_DIR}")

---

## Section 1 — Imitation

### Demonstrates Stage 1 (Imitation)


In [ ]:
# Build imitation env, load frozen teacher, prepare reference clips.
(
    im_env,
    im_reference_clips,
    im_teacher_fn,
    im_teacher_normalizer,
    im_teacher_cfg,
    im_distill_cfg,
    im_action_dist,
) = setup_imitation_env(
    IMITATION_DISTILL_CHECKPOINT,
    IMITATION_DISTILL_CHECKPOINT_STEP,
    teacher_checkpoint=IMITATION_TEACHER_CHECKPOINT_PATH,
    teacher_step=IMITATION_TEACHER_CHECKPOINT_STEP,
    mujoco_impl="warp",
    xml_variant_path=rodent_consts.RODENT_NO_TAIL_COLLISION_XML,
)

# Episode length = (clip_length - reference_length) * steps_per_frame.
_im_mocap_hz = float(im_distill_cfg.env_config.mocap_hz)
_im_steps_per_frame = int((1.0 / _im_mocap_hz) / im_distill_cfg.env_config.ctrl_dt)
IM_EPISODE_LENGTH = (
    im_distill_cfg.env_config.clip_length - im_distill_cfg.env_config.reference_length
) * _im_steps_per_frame
print(
    f"Imitation episode length: {IM_EPISODE_LENGTH} steps; clips: {IMITATION_NUM_CLIPS}"
)

#### Load rollouts

In [ ]:
_im_load_path = ROLLOUTS_DIR / "imitation_grid.h5"
_im_qpos, _im_attrs, _im_extras = load_rollout_h5(
    _im_load_path, extra_group_names=("qpos_ref",)
)
im_grid_qpos = stack_grid_qpos(_im_qpos, IMITATION_CLIP_INDICES)
im_grid_qpos_ref = stack_grid_qpos(_im_extras["qpos_ref"], IMITATION_CLIP_INDICES)
print(
    f"loaded im_grid_qpos {im_grid_qpos.shape}; im_grid_qpos_ref {im_grid_qpos_ref.shape}"
)

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# Roll out the frozen teacher across IMITATION_CLIP_INDICES (vmap'd JIT).
# def _im_teacher_action(obs):
#     flat_obs = flatten_obs_dict(obs)
#     norm_obs = normalize_dict_obs(flat_obs, im_teacher_normalizer)
#     logits = im_teacher_fn(norm_obs)
#     ctrl = im_action_dist.mode(logits)
#     return jnp.squeeze(ctrl, axis=0) if ctrl.ndim == 2 else ctrl


# def _im_step_fn(carry, _):
#     state, act_rng = carry
#     act_rng, _sub = jax.random.split(act_rng)
#     time_in_frames = state.data.time * _im_mocap_hz
#     frame = jnp.floor(time_in_frames + state.info["start_frame"]).astype(int)
#     ref = im_reference_clips.at(clip=state.info["reference_clip"], frame=frame)
#     ctrl = _im_teacher_action(state.obs)
#     next_state = im_env.step(state, ctrl)
#     return (next_state, act_rng), (state.data.qpos, ref.qpos)


# def _im_rollout_one(clip_idx):
#     rng = jax.random.PRNGKey(SEED)
#     reset_key, act_key = jax.random.split(rng)
#     state = im_env.reset(reset_key, clip_idx=clip_idx, start_frame=jnp.int32(0))
#     init_ref = im_reference_clips.at(clip=clip_idx, frame=jnp.int32(0))
#     (_, _), (qpos_seq, ref_seq) = jax.lax.scan(
#         _im_step_fn, (state, act_key), None, length=IM_EPISODE_LENGTH
#     )
#     qpos_full = jnp.concatenate([state.data.qpos[None], qpos_seq], axis=0)
#     ref_full = jnp.concatenate([init_ref.qpos[None], ref_seq], axis=0)
#     return {"qpos": qpos_full, "qpos_ref": ref_full}


# _im_clip_idxs = jnp.asarray(IMITATION_CLIP_INDICES, dtype=jnp.int32)
# print("JIT-compiling vmap rollout (slow first time)...")
# _im_rollout_vmap = jax.jit(jax.vmap(_im_rollout_one))
# _im_data = _im_rollout_vmap(_im_clip_idxs)
# im_grid_qpos = np.array(_im_data["qpos"])
# im_grid_qpos_ref = np.array(_im_data["qpos_ref"])
# print(
#     f"Done. im_grid_qpos {im_grid_qpos.shape}; im_grid_qpos_ref {im_grid_qpos_ref.shape}"
# )

# # Save imitation rollout (qpos + qpos_ref groups) to h5.
# _im_qpos_dict = {ci: im_grid_qpos[i] for i, ci in enumerate(IMITATION_CLIP_INDICES)}
# _im_ref_dict = {ci: im_grid_qpos_ref[i] for i, ci in enumerate(IMITATION_CLIP_INDICES)}
# save_rollout_h5(
#     ROLLOUTS_DIR / "imitation_grid.h5",
#     _im_qpos_dict,
#     attrs={
#         "kind": "imitation_teacher",
#         "clip_indices": np.asarray(IMITATION_CLIP_INDICES),
#         "episode_length": int(IM_EPISODE_LENGTH),
#         "distill_checkpoint": str(IMITATION_DISTILL_CHECKPOINT),
#         "seed": int(SEED),
#     },
#     extra_groups={"qpos_ref": _im_ref_dict},
# )

#### Render Video

In [ ]:
# Render the 3x3 grid: real rodents (opaque) + reference ghosts (transparent), each
# starting at its grid cell (xy is re-centered).
_im_model, _im_data_grid, _im_nq_single, _im_real_xy, _im_ghost_xy, _im_colors = (
    build_grid_scene(
        n_real=IMITATION_NUM_CLIPS, n_ghost=IMITATION_NUM_CLIPS, ghost_alpha=GHOST_ALPHA
    )
)
print(f"  grid model nq={_im_model.nq} (per-rodent nq={_im_nq_single})")

_im_real_q = recenter_to_grid(im_grid_qpos, _im_real_xy)
_im_ghost_q = recenter_to_grid(im_grid_qpos_ref, _im_ghost_xy)

_im_frames = render_grid_frames(
    _im_model,
    _im_data_grid,
    _im_nq_single,
    qpos_real=_im_real_q,
    qpos_ghost=_im_ghost_q,
)
_im_video = write_video(_im_frames, VIDEOS_DIR / "imitation_grid.mp4", fps=RENDER_FPS)
display(Video(str(_im_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "im_env",
    "im_reference_clips",
    "im_teacher_fn",
    "im_teacher_normalizer",
    "im_teacher_cfg",
    "im_distill_cfg",
    "im_action_dist",
    "im_grid_qpos",
    "im_grid_qpos_ref",
    "_im_rollout_vmap",
    "_im_data",
    "_im_clip_idxs",
    "_im_qpos_dict",
    "_im_ref_dict",
    "_im_model",
    "_im_data_grid",
    "_im_nq_single",
    "_im_real_xy",
    "_im_ghost_xy",
    "_im_colors",
    "_im_real_q",
    "_im_ghost_q",
    "_im_frames",
    "_im_video",
    "_im_qpos",
    "_im_attrs",
    "_im_extras",
)

---

## Section 2 — Random generation

### This section demonstrates the priors (behavior and motion) being rolled out using stochastic sampling.


## Behavior Prior

In [ ]:
# Load distill_motion_prior env + policy params, build behavior prior policy at logvar=-4.
(
    behavior_prior_env,
    behavior_prior_reference_clips,
    behavior_prior_cfg,
    behavior_prior_policy_params,
) = setup_distill_motion_prior_env(
    DISTILL_MOTION_PRIOR_CHECKPOINT, DISTILL_MOTION_PRIOR_CHECKPOINT_STEP
)
behavior_prior_policy_fn = create_behavior_prior_policy_fn(
    behavior_prior_policy_params,
    behavior_prior_cfg,
    fixed_logvar=-4.0,
    deterministic=False,
)
print("Behavior prior policy ready (logvar=-4)")

#### Load rollouts

In [ ]:
_behavior_prior_load_path = ROLLOUTS_DIR / "behavior_prior_logvar-4.h5"
_behavior_prior_qpos, _behavior_prior_attrs, _ = load_rollout_h5(
    _behavior_prior_load_path
)
behavior_prior_grid_qpos = stack_grid_qpos(_behavior_prior_qpos, CLIP_INDICES)
print(f"loaded behavior_prior_grid_qpos {behavior_prior_grid_qpos.shape}")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# Roll out behavior prior from each clip's frame-0 pose.
# _behavior_prior_qpos_by_clip = rollout_distill_motion_prior_grid(
#     behavior_prior_env,
#     behavior_prior_reference_clips,
#     behavior_prior_policy_fn,
#     CLIP_INDICES,
#     NUM_ROLLOUT_STEPS,
#     seed=SEED,
# )
# behavior_prior_grid_qpos = stack_grid_qpos(_behavior_prior_qpos_by_clip, CLIP_INDICES)
# print(f"behavior_prior_grid_qpos {behavior_prior_grid_qpos.shape}")

# save_rollout_h5(
#     ROLLOUTS_DIR / "behavior_prior_logvar-4.h5",
#     {ci: behavior_prior_grid_qpos[i] for i, ci in enumerate(CLIP_INDICES)},
#     attrs={
#         "kind": "behavior_prior",
#         "logvar": -4.0,
#         "clip_indices": np.asarray(CLIP_INDICES),
#         "num_steps": int(NUM_ROLLOUT_STEPS),
#         "distill_motion_prior_checkpoint": str(DISTILL_MOTION_PRIOR_CHECKPOINT),
#         "step": (
#             -1
#             if DISTILL_MOTION_PRIOR_CHECKPOINT_STEP is None
#             else int(DISTILL_MOTION_PRIOR_CHECKPOINT_STEP)
#         ),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
# Render the behavior prior 3x3 grid (no ghosts — just the rolled-out rodents).
(
    _behavior_prior_model,
    _behavior_prior_data_grid,
    _behavior_prior_nq_single,
    _behavior_prior_real_xy,
    _,
    _,
) = build_grid_scene(n_real=NUM_CLIPS, n_ghost=0)
print(
    f"  grid model nq={_behavior_prior_model.nq} (per-rodent nq={_behavior_prior_nq_single})"
)
_behavior_prior_real_q = recenter_to_grid(
    behavior_prior_grid_qpos, _behavior_prior_real_xy
)
_behavior_prior_frames = render_grid_frames(
    _behavior_prior_model,
    _behavior_prior_data_grid,
    _behavior_prior_nq_single,
    qpos_real=_behavior_prior_real_q,
)
_behavior_prior_video = write_video(
    _behavior_prior_frames, VIDEOS_DIR / "behavior_prior_logvar-4.mp4", fps=RENDER_FPS
)
display(Video(str(_behavior_prior_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "behavior_prior_env",
    "behavior_prior_reference_clips",
    "behavior_prior_cfg",
    "behavior_prior_policy_params",
    "behavior_prior_policy_fn",
    "behavior_prior_grid_qpos",
    "_behavior_prior_qpos_by_clip",
    "_behavior_prior_model",
    "_behavior_prior_data_grid",
    "_behavior_prior_nq_single",
    "_behavior_prior_real_xy",
    "_behavior_prior_real_q",
    "_behavior_prior_frames",
    "_behavior_prior_video",
    "_behavior_prior_qpos",
    "_behavior_prior_attrs",
)

## Motion Prior (low noise)

In [ ]:
(
    motion_prior4_env,
    motion_prior4_reference_clips,
    motion_prior4_cfg,
    motion_prior4_policy_params,
) = setup_distill_motion_prior_env(
    DISTILL_MOTION_PRIOR_CHECKPOINT, DISTILL_MOTION_PRIOR_CHECKPOINT_STEP
)
motion_prior4_policy_fn = create_motion_prior_policy_fn(
    motion_prior4_policy_params,
    motion_prior4_cfg,
    fixed_logvar=-4.0,
    deterministic=False,
)
print("Motion prior policy ready (logvar=-4)")

#### Load rollouts

In [ ]:
_p4_load_path = ROLLOUTS_DIR / "motion_prior_logvar-4.h5"
_p4_qpos, _p4_attrs, _ = load_rollout_h5(_p4_load_path)
motion_prior4_grid_qpos = stack_grid_qpos(_p4_qpos, CLIP_INDICES)
print(f"loaded motion_prior4_grid_qpos {motion_prior4_grid_qpos.shape}")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# _p4_qpos_by_clip = rollout_distill_motion_prior_grid(
#     motion_prior4_env,
#     motion_prior4_reference_clips,
#     motion_prior4_policy_fn,
#     CLIP_INDICES,
#     NUM_ROLLOUT_STEPS,
#     seed=SEED,
# )
# motion_prior4_grid_qpos = stack_grid_qpos(_p4_qpos_by_clip, CLIP_INDICES)
# print(f"motion_prior4_grid_qpos {motion_prior4_grid_qpos.shape}")

# save_rollout_h5(
#     ROLLOUTS_DIR / "motion_prior_logvar-4.h5",
#     {ci: motion_prior4_grid_qpos[i] for i, ci in enumerate(CLIP_INDICES)},
#     attrs={
#         "kind": "motion_prior",
#         "logvar": -4.0,
#         "clip_indices": np.asarray(CLIP_INDICES),
#         "num_steps": int(NUM_ROLLOUT_STEPS),
#         "distill_motion_prior_checkpoint": str(DISTILL_MOTION_PRIOR_CHECKPOINT),
#         "step": (
#             -1
#             if DISTILL_MOTION_PRIOR_CHECKPOINT_STEP is None
#             else int(DISTILL_MOTION_PRIOR_CHECKPOINT_STEP)
#         ),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
_p4_model, _p4_data_grid, _p4_nq_single, _p4_real_xy, _, _ = build_grid_scene(
    n_real=NUM_CLIPS, n_ghost=0
)
print(f"  grid model nq={_p4_model.nq} (per-rodent nq={_p4_nq_single})")
_p4_real_q = recenter_to_grid(motion_prior4_grid_qpos, _p4_real_xy)
_p4_frames = render_grid_frames(
    _p4_model, _p4_data_grid, _p4_nq_single, qpos_real=_p4_real_q
)
_p4_video = write_video(
    _p4_frames, VIDEOS_DIR / "motion_prior_logvar-4.mp4", fps=RENDER_FPS
)
display(Video(str(_p4_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "motion_prior4_env",
    "motion_prior4_reference_clips",
    "motion_prior4_cfg",
    "motion_prior4_policy_params",
    "motion_prior4_policy_fn",
    "motion_prior4_grid_qpos",
    "_p4_qpos_by_clip",
    "_p4_model",
    "_p4_data_grid",
    "_p4_nq_single",
    "_p4_real_xy",
    "_p4_real_q",
    "_p4_frames",
    "_p4_video",
    "_p4_qpos",
    "_p4_attrs",
)

## Motion Prior (high noise)


In [ ]:
(
    motion_prior0_env,
    motion_prior0_reference_clips,
    motion_prior0_cfg,
    motion_prior0_policy_params,
) = setup_distill_motion_prior_env(
    DISTILL_MOTION_PRIOR_CHECKPOINT, DISTILL_MOTION_PRIOR_CHECKPOINT_STEP
)
motion_prior0_policy_fn = create_motion_prior_policy_fn(
    motion_prior0_policy_params,
    motion_prior0_cfg,
    fixed_logvar=0.0,
    deterministic=False,
)
print("Motion prior policy ready (logvar=0)")

#### Load rollouts

In [ ]:
_p0_load_path = ROLLOUTS_DIR / "motion_prior_logvar0.h5"
_p0_qpos, _p0_attrs, _ = load_rollout_h5(_p0_load_path)
motion_prior0_grid_qpos = stack_grid_qpos(_p0_qpos, CLIP_INDICES)
print(f"loaded motion_prior0_grid_qpos {motion_prior0_grid_qpos.shape}")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# _p0_qpos_by_clip = rollout_distill_motion_prior_grid(
#     motion_prior0_env,
#     motion_prior0_reference_clips,
#     motion_prior0_policy_fn,
#     CLIP_INDICES,
#     NUM_ROLLOUT_STEPS,
#     seed=SEED,
# )
# motion_prior0_grid_qpos = stack_grid_qpos(_p0_qpos_by_clip, CLIP_INDICES)
# print(f"motion_prior0_grid_qpos {motion_prior0_grid_qpos.shape}")

# save_rollout_h5(
#     ROLLOUTS_DIR / "motion_prior_logvar0.h5",
#     {ci: motion_prior0_grid_qpos[i] for i, ci in enumerate(CLIP_INDICES)},
#     attrs={
#         "kind": "motion_prior",
#         "logvar": 0.0,
#         "clip_indices": np.asarray(CLIP_INDICES),
#         "num_steps": int(NUM_ROLLOUT_STEPS),
#         "distill_motion_prior_checkpoint": str(DISTILL_MOTION_PRIOR_CHECKPOINT),
#         "step": (
#             -1
#             if DISTILL_MOTION_PRIOR_CHECKPOINT_STEP is None
#             else int(DISTILL_MOTION_PRIOR_CHECKPOINT_STEP)
#         ),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
_p0_model, _p0_data_grid, _p0_nq_single, _p0_real_xy, _, _ = build_grid_scene(
    n_real=NUM_CLIPS, n_ghost=0
)
print(f"  grid model nq={_p0_model.nq} (per-rodent nq={_p0_nq_single})")
_p0_real_q = recenter_to_grid(motion_prior0_grid_qpos, _p0_real_xy)
_p0_frames = render_grid_frames(
    _p0_model, _p0_data_grid, _p0_nq_single, qpos_real=_p0_real_q
)
_p0_video = write_video(
    _p0_frames, VIDEOS_DIR / "motion_prior_logvar0.mp4", fps=RENDER_FPS
)
display(Video(str(_p0_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "motion_prior0_env",
    "motion_prior0_reference_clips",
    "motion_prior0_cfg",
    "motion_prior0_policy_params",
    "motion_prior0_policy_fn",
    "motion_prior0_grid_qpos",
    "_p0_qpos_by_clip",
    "_p0_model",
    "_p0_data_grid",
    "_p0_nq_single",
    "_p0_real_xy",
    "_p0_real_q",
    "_p0_frames",
    "_p0_video",
    "_p0_qpos",
    "_p0_attrs",
)

---

## Section 3 — Walk-rear

### This section demonstrates the trained models being rolled out on the walk-then-rear task

## SCAMPER

In [ ]:
(
    wr_env,
    wr_base_env,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    wr_start_state,
    wr_config,
    wr_mode,
) = setup_walkrear(WALKREAR_SCAMPER_CHECKPOINT, WALKREAR_SCAMPER_STEP)
print(f"  obs_size={wr_start_state.obs.shape[-1]}, action_size={wr_env.action_size}")

#### Load rollouts

In [ ]:
_wr_qpos_dict, _wr_attrs, _ = load_rollout_h5(ROLLOUTS_DIR / "walkrear_scamper.h5")
wr_qpos = np.asarray(list(_wr_qpos_dict.values())[0])
print(f"loaded wr_qpos {wr_qpos.shape}")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# wr_qpos, wr_states = task_transfer_rollout(
#     wr_start_state,
#     wr_jit_step,
#     wr_jit_inf,
#     wr_inf_params,
#     WALKREAR_NUM_STEPS,
#     seed=SEED,
# )
# print(
#     f"wr_qpos {wr_qpos.shape}; final torso xy={wr_qpos[-1, :2]}, z={wr_qpos[-1, 2]:.3f}"
# )

# save_rollout_h5(
#     ROLLOUTS_DIR / "walkrear_scamper.h5",
#     {0: wr_qpos},
#     attrs={
#         "kind": "walkrear",
#         "variant": "SCAMPER",
#         "mode": str(wr_mode),
#         "checkpoint": str(WALKREAR_SCAMPER_CHECKPOINT),
#         "step": -1 if WALKREAR_SCAMPER_STEP is None else int(WALKREAR_SCAMPER_STEP),
#         "num_steps_requested": int(WALKREAR_NUM_STEPS),
#         "num_steps_actual": int(wr_qpos.shape[0]),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
_wr_frames = render_walkrear_qpos(wr_base_env, wr_qpos)
_wr_video = write_video(_wr_frames, VIDEOS_DIR / "walkrear_scamper.mp4", fps=RENDER_FPS)
display(Video(str(_wr_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "wr_env",
    "wr_base_env",
    "wr_jit_step",
    "wr_jit_inf",
    "wr_inf_params",
    "wr_start_state",
    "wr_config",
    "wr_mode",
    "wr_qpos",
    "wr_states",
    "_wr_frames",
    "_wr_video",
    "_wr_qpos_dict",
    "_wr_attrs",
)

## MIMIC-MJX


In [ ]:
(
    wr_env,
    wr_base_env,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    wr_start_state,
    wr_config,
    wr_mode,
) = setup_walkrear(WALKREAR_MIMICMJX_CHECKPOINT, WALKREAR_MIMICMJX_STEP)

#### Load rollouts

In [ ]:
_wr_qpos_dict, _wr_attrs, _ = load_rollout_h5(ROLLOUTS_DIR / "walkrear_mimicmjx.h5")
wr_qpos = np.asarray(list(_wr_qpos_dict.values())[0])
print(f"loaded wr_qpos {wr_qpos.shape}")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# wr_qpos, wr_states = task_transfer_rollout(
#     wr_start_state,
#     wr_jit_step,
#     wr_jit_inf,
#     wr_inf_params,
#     WALKREAR_NUM_STEPS,
#     seed=SEED,
# )
# print(
#     f"wr_qpos {wr_qpos.shape}; final torso xy={wr_qpos[-1, :2]}, z={wr_qpos[-1, 2]:.3f}"
# )

# save_rollout_h5(
#     ROLLOUTS_DIR / "walkrear_mimicmjx.h5",
#     {0: wr_qpos},
#     attrs={
#         "kind": "walkrear",
#         "variant": "MIMIC-MJX",
#         "mode": str(wr_mode),
#         "checkpoint": str(WALKREAR_MIMICMJX_CHECKPOINT),
#         "step": -1 if WALKREAR_MIMICMJX_STEP is None else int(WALKREAR_MIMICMJX_STEP),
#         "num_steps_requested": int(WALKREAR_NUM_STEPS),
#         "num_steps_actual": int(wr_qpos.shape[0]),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
_wr_frames = render_walkrear_qpos(wr_base_env, wr_qpos)
_wr_video = write_video(
    _wr_frames, VIDEOS_DIR / "walkrear_mimicmjx.mp4", fps=RENDER_FPS
)
display(Video(str(_wr_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "wr_env",
    "wr_base_env",
    "wr_jit_step",
    "wr_jit_inf",
    "wr_inf_params",
    "wr_start_state",
    "wr_config",
    "wr_mode",
    "wr_qpos",
    "wr_states",
    "_wr_frames",
    "_wr_video",
    "_wr_qpos_dict",
    "_wr_attrs",
)

## From-Scratch Model


In [ ]:
(
    wr_env,
    wr_base_env,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    wr_start_state,
    wr_config,
    wr_mode,
) = setup_walkrear(WALKREAR_SCRATCH_CHECKPOINT, WALKREAR_SCRATCH_STEP)

#### Load rollouts

In [ ]:
_wr_qpos_dict, _wr_attrs, _ = load_rollout_h5(ROLLOUTS_DIR / "walkrear_scratch.h5")
wr_qpos = np.asarray(list(_wr_qpos_dict.values())[0])
print(f"loaded wr_qpos {wr_qpos.shape}")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# wr_qpos, wr_states = task_transfer_rollout(
#     wr_start_state,
#     wr_jit_step,
#     wr_jit_inf,
#     wr_inf_params,
#     WALKREAR_NUM_STEPS,
#     seed=SEED,
# )
# print(
#     f"wr_qpos {wr_qpos.shape}; final torso xy={wr_qpos[-1, :2]}, z={wr_qpos[-1, 2]:.3f}"
# )

# save_rollout_h5(
#     ROLLOUTS_DIR / "walkrear_scratch.h5",
#     {0: wr_qpos},
#     attrs={
#         "kind": "walkrear",
#         "variant": "scratch",
#         "mode": str(wr_mode),
#         "checkpoint": str(WALKREAR_SCRATCH_CHECKPOINT),
#         "step": -1 if WALKREAR_SCRATCH_STEP is None else int(WALKREAR_SCRATCH_STEP),
#         "num_steps_requested": int(WALKREAR_NUM_STEPS),
#         "num_steps_actual": int(wr_qpos.shape[0]),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
_wr_frames = render_walkrear_qpos(wr_base_env, wr_qpos)
_wr_video = write_video(_wr_frames, VIDEOS_DIR / "walkrear_scratch.mp4", fps=RENDER_FPS)
display(Video(str(_wr_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "wr_env",
    "wr_base_env",
    "wr_jit_step",
    "wr_jit_inf",
    "wr_inf_params",
    "wr_start_state",
    "wr_config",
    "wr_mode",
    "wr_qpos",
    "wr_states",
    "_wr_frames",
    "_wr_video",
    "_wr_qpos_dict",
    "_wr_attrs",
)

---

## Section 4 — Arena-forage

### This section demonstrates the models being rolled out on the arena foraging task


## SCAMPER

In [ ]:
(
    af_env_,
    af_base_env,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    af_start_state,
    af_config,
    af_mode,
) = setup_arena_forage(ARENAFORAGE_SCAMPER_CHECKPOINT, ARENAFORAGE_SCAMPER_STEP)
print(f"  obs_size={af_start_state.obs.shape[-1]}, action_size={af_env_.action_size}")

#### Load rollouts

In [ ]:
af_rollout, af_attrs = load_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_scamper.h5"
)
print(
    f"loaded qpos {af_rollout['qpos'].shape}, treat_collected {af_rollout['treat_collected'].shape}"
)

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# af_rollout, af_states = arena_forage_rollout(
#     af_start_state,
#     af_jit_step,
#     af_jit_inf,
#     af_inf_params,
#     ARENAFORAGE_NUM_STEPS,
#     seed=SEED,
# )
# print(
#     f"qpos {af_rollout['qpos'].shape}; treats collected at end: "
#     f"{int(af_rollout['treat_collected'][-1].sum())}/{af_rollout['treat_collected'].shape[1]}"
# )

# save_arena_forage_rollout(
#     ROLLOUTS_DIR / "arenaforage_scamper.h5",
#     af_rollout,
#     attrs={
#         "kind": "arenaforage",
#         "variant": "SCAMPER",
#         "mode": str(af_mode),
#         "checkpoint": str(ARENAFORAGE_SCAMPER_CHECKPOINT),
#         "step": (
#             -1 if ARENAFORAGE_SCAMPER_STEP is None else int(ARENAFORAGE_SCAMPER_STEP)
#         ),
#         "num_steps_requested": int(ARENAFORAGE_NUM_STEPS),
#         "num_steps_actual": int(af_rollout["qpos"].shape[0]),
#         "scent_sigma": float(af_base_env._config.scent_sigma),
#         "spawn_radius": float(af_base_env._config.spawn_radius),
#         "num_treats": int(af_base_env._config.num_treats),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
_af_video = render_arena_forage_3panel(
    af_base_env,
    af_states,
    af_mode,
    mode_config=af_config.get(af_mode, {}),
    out_path=VIDEOS_DIR / "arenaforage_scamper.mp4",
)
display(Video(str(_af_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "af_env_",
    "af_base_env",
    "af_jit_step",
    "af_jit_inf",
    "af_inf_params",
    "af_start_state",
    "af_config",
    "af_mode",
    "af_rollout",
    "af_states",
    "af_attrs",
    "_af_video",
)

## MIMIC-MJX


In [ ]:
(
    af_env_,
    af_base_env,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    af_start_state,
    af_config,
    af_mode,
) = setup_arena_forage(ARENAFORAGE_MIMICMJX_CHECKPOINT, ARENAFORAGE_MIMICMJX_STEP)

#### Load rollouts

In [ ]:
af_rollout, af_attrs = load_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_mimicmjx.h5"
)
print(f"loaded qpos {af_rollout['qpos'].shape}")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# af_rollout, af_states = arena_forage_rollout(
#     af_start_state,
#     af_jit_step,
#     af_jit_inf,
#     af_inf_params,
#     ARENAFORAGE_NUM_STEPS,
#     seed=SEED,
# )
# print(
#     f"qpos {af_rollout['qpos'].shape}; treats collected at end: "
#     f"{int(af_rollout['treat_collected'][-1].sum())}/{af_rollout['treat_collected'].shape[1]}"
# )

# save_arena_forage_rollout(
#     ROLLOUTS_DIR / "arenaforage_mimicmjx.h5",
#     af_rollout,
#     attrs={
#         "kind": "arenaforage",
#         "variant": "MIMIC-MJX",
#         "mode": str(af_mode),
#         "checkpoint": str(ARENAFORAGE_MIMICMJX_CHECKPOINT),
#         "step": (
#             -1 if ARENAFORAGE_MIMICMJX_STEP is None else int(ARENAFORAGE_MIMICMJX_STEP)
#         ),
#         "num_steps_requested": int(ARENAFORAGE_NUM_STEPS),
#         "num_steps_actual": int(af_rollout["qpos"].shape[0]),
#         "scent_sigma": float(af_base_env._config.scent_sigma),
#         "spawn_radius": float(af_base_env._config.spawn_radius),
#         "num_treats": int(af_base_env._config.num_treats),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
_af_video = render_arena_forage_3panel(
    af_base_env,
    af_states,
    af_mode,
    mode_config=af_config.get(af_mode, {}),
    out_path=VIDEOS_DIR / "arenaforage_mimicmjx.mp4",
)
display(Video(str(_af_video), embed=True))

#### Garbage collection

In [ ]:
release(
    "af_env_",
    "af_base_env",
    "af_jit_step",
    "af_jit_inf",
    "af_inf_params",
    "af_start_state",
    "af_config",
    "af_mode",
    "af_rollout",
    "af_states",
    "af_attrs",
    "_af_video",
)

## From-Scratch model


In [ ]:
(
    af_env_,
    af_base_env,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    af_start_state,
    af_config,
    af_mode,
) = setup_arena_forage(ARENAFORAGE_SCRATCH_CHECKPOINT, ARENAFORAGE_SCRATCH_STEP)

#### Load rollouts

In [ ]:
af_rollout, af_attrs = load_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_scratch.h5"
)
print(f"loaded qpos {af_rollout['qpos'].shape}")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# af_rollout, af_states = arena_forage_rollout(
#     af_start_state,
#     af_jit_step,
#     af_jit_inf,
#     af_inf_params,
#     ARENAFORAGE_NUM_STEPS,
#     seed=SEED,
# )
# print(
#     f"qpos {af_rollout['qpos'].shape}; treats collected at end: "
#     f"{int(af_rollout['treat_collected'][-1].sum())}/{af_rollout['treat_collected'].shape[1]}"
# )

# save_arena_forage_rollout(
#     ROLLOUTS_DIR / "arenaforage_scratch.h5",
#     af_rollout,
#     attrs={
#         "kind": "arenaforage",
#         "variant": "scratch",
#         "mode": str(af_mode),
#         "checkpoint": str(ARENAFORAGE_SCRATCH_CHECKPOINT),
#         "step": (
#             -1 if ARENAFORAGE_SCRATCH_STEP is None else int(ARENAFORAGE_SCRATCH_STEP)
#         ),
#         "num_steps_requested": int(ARENAFORAGE_NUM_STEPS),
#         "num_steps_actual": int(af_rollout["qpos"].shape[0]),
#         "scent_sigma": float(af_base_env._config.scent_sigma),
#         "spawn_radius": float(af_base_env._config.spawn_radius),
#         "num_treats": int(af_base_env._config.num_treats),
#         "seed": int(SEED),
#     },
# )

#### Render video

In [ ]:
_af_video = render_arena_forage_3panel(
    af_base_env,
    af_states,
    af_mode,
    mode_config=af_config.get(af_mode, {}),
    out_path=VIDEOS_DIR / "arenaforage_scratch.mp4",
)
display(Video(str(_af_video), embed=True))

In [ ]:
release(
    "af_env_",
    "af_base_env",
    "af_jit_step",
    "af_jit_inf",
    "af_inf_params",
    "af_start_state",
    "af_config",
    "af_mode",
    "af_rollout",
    "af_states",
    "af_attrs",
    "_af_video",
)

---

## Section 5 — Locomotion Task (Maintain velocity)

### This section demonstrates the models being rolled out on the locomotion task

In [ ]:
# Per-config maintain-velocity sweep (3 modes x 3 speeds). Sensor thresholds and
# gait-analysis windows are tuned per checkpoint to match gait_analysis_combined.
MAINTAIN_VELOCITY_CONFIGS = [
    {
        "name": "SCAMPER_S3e-1",
        "method": "SCAMPER",
        "gait": "walk",
        "checkpoint_path": "../data/checkpoints/locomotion/scamper_s3e-1/260403_015403_017303",
        "checkpoint_step": 25,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 0.5,
        "touch_threshold_sole_r": 0.5,
        "min_stance_gap": 10,
    },
    {
        "name": "MIMIC-MJX_S3e-1",
        "method": "MIMIC-MJX",
        "gait": "walk",
        "checkpoint_path": "../data/checkpoints/locomotion/mimic-mjx_s3e-1/260226_085923_279627",
        "checkpoint_step": 25,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 0.5,
        "touch_threshold_sole_r": 0.5,
        "min_stance_gap": 10,
    },
    {
        "name": "Scratch_S3e-1",
        "method": "Scratch",
        "gait": "walk",
        "checkpoint_path": "../data/checkpoints/locomotion/scratch_s3e-1/260228_120947_605589",
        "checkpoint_step": 25,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 0.5,
        "touch_threshold_sole_r": 0.5,
        "min_stance_gap": 10,
    },
    {
        "name": "SCAMPER_S5e-1",
        "method": "SCAMPER",
        "gait": "trot",
        "checkpoint_path": "../data/checkpoints/locomotion/scamper_s5e-1/260403_054936_177691",
        "checkpoint_step": 24,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 0.5,
        "touch_threshold_sole_r": 0.5,
        "min_stance_gap": 10,
        "spike_removal_sensor": "palm_L",
        "spike_threshold": 0.01,
        "spike_max_len": 8,
    },
    {
        "name": "MIMIC-MJX_S5e-1",
        "method": "MIMIC-MJX",
        "gait": "trot",
        "checkpoint_path": "../data/checkpoints/locomotion/mimic-mjx_s5e-1/260226_090851_633008",
        "checkpoint_step": 23,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 0.5,
        "touch_threshold_sole_r": 0.5,
        "min_stance_gap": 6,
    },
    {
        "name": "Scratch_S5e-1",
        "method": "Scratch",
        "gait": "trot",
        "checkpoint_path": "../data/checkpoints/locomotion/scratch_s5e-1/260228_120949_207733",
        "checkpoint_step": 25,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 0.5,
        "touch_threshold_sole_r": 0.5,
        "min_stance_gap": 10,
    },
    {
        "name": "SCAMPER_S7e-1",
        "method": "SCAMPER",
        "gait": "gallop",
        "checkpoint_path": "../data/checkpoints/locomotion/scamper_s7e-1/260403_054936_245874",
        "checkpoint_step": 23,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 1.0,
        "touch_threshold_sole_r": 1.0,
        "min_stance_gap": 6,
    },
    {
        "name": "MIMIC-MJX_S7e-1",
        "method": "MIMIC-MJX",
        "gait": "gallop",
        "checkpoint_path": "../data/checkpoints/locomotion/mimic-mjx_s7e-1/260226_090849_117360",
        "checkpoint_step": 23,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 1.0,
        "touch_threshold_sole_r": 1.0,
        "min_stance_gap": 6,
    },
    {
        "name": "Scratch_S7e-1",
        "method": "Scratch",
        "gait": "gallop",
        "checkpoint_path": "../data/checkpoints/locomotion/scratch_s7e-1/260228_115525_352293",
        "checkpoint_step": 25,
        "seed": 0,
        "episode_length": 1000,
        "touch_threshold_palm_l": 0.01,
        "touch_threshold_palm_r": 0.01,
        "touch_threshold_sole_l": 1.0,
        "touch_threshold_sole_r": 1.0,
        "min_stance_gap": 6,
    },
]

MV_ROLLOUTS_DIR = ROLLOUTS_DIR / "locomotion"
MV_VIDEOS_DIR = VIDEOS_DIR / "locomotion"
MV_ROLLOUTS_DIR.mkdir(parents=True, exist_ok=True)
MV_VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(MAINTAIN_VELOCITY_CONFIGS)} maintain-velocity configs registered")

#### Load rollouts

In [ ]:
# Load all 9 cached rollouts (regenerate via the ROLLOUT cell below if missing).
mv_rollouts = {}
for cfg in MAINTAIN_VELOCITY_CONFIGS:
    path = MV_ROLLOUTS_DIR / f"{cfg['name']}.h5"
    if not path.exists():
        print(f"  missing {path} — run the ROLLOUT cell to generate it")
        continue
    arrays, attrs = load_mv_rollout_h5(path)
    mv_rollouts[cfg["name"]] = arrays
    print(f"  loaded {cfg['name']} ({arrays['qpos'].shape[0]} steps)")

#### ROLLOUT CELL (SKIP IF NO GPU)

In [ ]:
# ROLLOUT (UNCOMMENT IF YOU WANT TO PERFORM YOUR OWN ROLLOUTS)
# Skip this cell if you only want to render from the cached h5 files above.
# mv_rollouts = {}
# for cfg in MAINTAIN_VELOCITY_CONFIGS:
#     name = cfg["name"]
#     print(f"\n{'=' * 60}\nRolling out {name}\n{'=' * 60}")
#     (
#         mv_env,
#         mv_base_env,
#         mv_jit_reset,
#         mv_jit_step,
#         mv_jit_inf,
#         mv_inf_params,
#         mv_config,
#         mv_ckpt_cfg,
#         mv_mode,
#     ) = setup_maintain_velocity(cfg["checkpoint_path"], cfg["checkpoint_step"])
#     arrays, _ = maintain_velocity_rollout(
#         mv_env,
#         mv_jit_reset,
#         mv_jit_step,
#         mv_jit_inf,
#         mv_inf_params,
#         num_steps=cfg["episode_length"],
#         seed=cfg["seed"],
#     )
#     save_mv_rollout_h5(
#         MV_ROLLOUTS_DIR / f"{name}.h5",
#         arrays,
#         attrs={
#             "kind": "maintain_velocity",
#             "name": name,
#             "method": cfg["method"],
#             "gait": cfg["gait"],
#             "mode": str(mv_mode),
#             "checkpoint": str(cfg["checkpoint_path"]),
#             "step": int(cfg["checkpoint_step"]),
#             "num_steps": int(arrays["qpos"].shape[0]),
#             "seed": int(cfg["seed"]),
#         },
#     )
#     mv_rollouts[name] = arrays
#     release(
#         "mv_env",
#         "mv_base_env",
#         "mv_jit_reset",
#         "mv_jit_step",
#         "mv_jit_inf",
#         "mv_inf_params",
#         "mv_config",
#         "mv_ckpt_cfg",
#         "mv_mode",
#     )

#### Render videos

In [63]:
# Render the composite swing/stance video for each config. One env is rebuilt
# per checkpoint because the wrapper's .render uses its own mj_model. Released
# between configs to bound peak memory.
for cfg in MAINTAIN_VELOCITY_CONFIGS:
    name = cfg["name"]
    if name not in mv_rollouts:
        print(f"  skipping {name} (no cached rollout)")
        continue
    print(f"\n{'=' * 60}\nRendering {name}\n{'=' * 60}")
    arrays = mv_rollouts[name]
    (
        mv_env,
        mv_base_env,
        _,
        _,
        _,
        _,
        _,
        _,
        _,
    ) = setup_maintain_velocity(cfg["checkpoint_path"], cfg["checkpoint_step"])
    palm_l, palm_r, sole_l, sole_r = mv_extract_touch_sensors(
        arrays["sensordata"], mv_base_env.mj_model
    )
    contact_barcode = mv_compute_contact_barcode(palm_l, palm_r, sole_l, sole_r, cfg)
    video_path = render_mv_composite(
        mv_env,
        arrays,
        contact_barcode,
        out_path=MV_VIDEOS_DIR / f"{name}_with_gait.mp4",
    )
    display(Video(str(video_path), embed=True))
    release("mv_env", "mv_base_env")

#### Garbage collection

In [ ]:
release("mv_rollouts")